In [175]:
import numpy as np
import pandas as pd
import os
from helper_files.metrics import average_pop

In [16]:
experiments_folder = 'experiments'
experiment_name = 'babyLFM5k'

In [17]:
tracks = pd.read_csv(f'{experiments_folder}/{experiment_name}/input/tracks.tsv', sep="\t", header=None, names=["track_id","artist","title","country","gender"])

In [18]:
dataset = pd.read_csv(f'{experiments_folder}/{experiment_name}/input/dataset.inter', sep="\t", header=0)

In [91]:
demographics = pd.read_csv(f'{experiments_folder}/{experiment_name}/input/demographics.tsv', sep="\t", header=None, names=["country","age","gender","user_id"])

In [19]:
tracks.head()

,track_id,artist,title,country,gender
0,769,Miguel,...All,ES,Male
1,258,Britney Spears,...Baby One More Time,US,Female
2,729,J. Cole,03' Adolescence,US,Male
3,481,Miley Cyrus,1 Sun,US,Non-binary
4,2007,Amerie,1 Thing,US,Female


In [20]:
dataset.head()

,user_id:token,item_id:token
0,870,0
1,534,1
2,390,2
3,218,3
4,564,4


In [22]:
artists_popularity = tracks.merge(dataset, left_on="track_id", right_on="item_id:token")["artist"].value_counts()/len(dataset)

In [23]:
artist_exposures = tracks[["artist","country","gender"]].drop_duplicates()
artist_exposures = artist_exposures.merge(artists_popularity, left_on="artist", right_index=True, how="left").rename(columns={"count":"train_popularity"})

In [34]:
# find num_iter by finding the highest number from the log folders


log_folders = [f for f in os.listdir(f'{experiments_folder}/{experiment_name}/log') if os.path.isdir(os.path.join(f'{experiments_folder}/{experiment_name}/log', f))]
num_iter = max([int(f.split('_')[-1]) for f in log_folders])

In [70]:
for i in np.arange(1,num_iter+1):
    top_k = pd.read_csv(f"experiments/babyLFM5k/output/iteration_{i}_top_k.tsv", sep="\t", header=0)
    top_k["exposure"] = 1/np.log2(1 + top_k["rank"])
    ndcg = pd.read_csv(f"{experiments_folder}/{experiment_name}/output/iteration_{i}_ndcg_per_user.tsv", sep="\t", header=0)

    iteration_exposure = tracks.merge(top_k, left_on="track_id", right_on="item_id")[["artist","exposure"]].groupby("artist").sum()
    iteration_exposure = iteration_exposure / iteration_exposure.sum()

    artist_exposures = artist_exposures.merge(iteration_exposure, left_on="artist", right_index=True, how="left").rename(columns={"exposure":f"iteration_{i}_exposure"}).fillna(0)

    

    



# RG2.1

In [ ]:
for i in np.arange(1,num_iter+1):
    top_k = pd.read_csv(f"experiments/babyLFM5k/output/iteration_{i}_top_k.tsv", sep="\t", header=0)
    top_k["exposure"] = 1/np.log2(1 + top_k["rank"])

    accepted_songs = top_k[["user_id","rank","exposure"]]
    accepted_songs["artist"] = top_k.merge(tracks[["track_id","artist"]], left_on="item_id", right_on="track_id")["artist"]
    accepted_songs = accepted_songs.merge(demographics, left_on="user_id", right_on="user_id", how="left")
    accepted_songs.rename(columns={"country":"user_country","gender":"user_gender","age":"user_age"}, inplace=True)

    accepted_songs = accepted_songs.merge(tracks[["artist","country","gender"]].drop_duplicates(), left_on="artist", right_on="artist", how="left").rename(columns={"country":"artist_country","gender":"artist_gender"})

    accepted_songs.to_csv(f"{experiments_folder}/{experiment_name}/accepted_songs_iteration_{i}.csv", index=False)




# RG2.2

In [176]:
user_statistics_all_iters = demographics.rename(columns={"country":"user_country","gender":"user_gender","age":"user_age"})
artist_exposures = tracks[["artist","country","gender"]].drop_duplicates()
artist_exposures = artist_exposures.merge(artists_popularity, left_on="artist", right_index=True, how="left").rename(columns={"count":"train_popularity"})


for i in np.arange(1,num_iter+1):
        top_k = pd.read_csv(f"{experiments_folder}/{experiment_name}/output/iteration_{i}_top_k.tsv", sep="\t", header=0)
        top_k["exposure"] = 1/np.log2(1 + top_k["rank"])
        ndcg = pd.read_csv(f"{experiments_folder}/{experiment_name}/output/iteration_{i}_ndcg_per_user.tsv", sep="\t", header=0)

        iteration_exposure = tracks.merge(top_k, left_on="track_id", right_on="item_id")[["artist","exposure"]].groupby("artist").sum()
        iteration_exposure = iteration_exposure / iteration_exposure.sum()

        artist_exposures = artist_exposures.merge(iteration_exposure, left_on="artist", right_index=True, how="left").rename(columns={"exposure":f"iteration_{i}_exposure"}).fillna(0)

        # for RG2.1
        accepted_songs = top_k[["user_id","rank","exposure"]]
        accepted_songs["artist"] = top_k.merge(tracks[["track_id","artist"]], left_on="item_id", right_on="track_id")["artist"]
        accepted_songs = accepted_songs.merge(demographics, left_on="user_id", right_on="user_id", how="left")
        accepted_songs.rename(columns={"country":"user_country","gender":"user_gender","age":"user_age"}, inplace=True)

        accepted_songs = accepted_songs.merge(tracks[["artist","country","gender"]].drop_duplicates(), left_on="artist", right_on="artist", how="left").rename(columns={"country":"artist_country","gender":"artist_gender"})

        # for RG2.2 
        user_statistics_all_iters = user_statistics_all_iters.merge(ndcg, left_on="user_id", right_on="user_id", how="left").rename(columns={"ndcg@10":f"iteration_{i}_ndcg@10"})
        pop = average_pop(accepted_songs, artist_exposures)

        user_statistics_all_iters[f"iteration_{i}_average_pop"] = pop

/tmp/ipykernel_3670991/3344124924.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  accepted_songs["artist"] = top_k.merge(tracks[["track_id","artist"]], left_on="item_id", right_on="track_id")["artist"]
/tmp/ipykernel_3670991/3344124924.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  accepted_songs["artist"] = top_k.merge(tracks[["track_id","artist"]], left_on="item_id", right_on="track_id")["artist"]
/tmp/ipykernel_3670991/3344124924.py:13: SettingWithCopyWarning: 
A value is trying to be set on a

In [188]:
dataset_artists = tracks.merge(dataset, left_on="track_id", right_on="item_id:token")[["user_id:token","artist"]].rename(columns={"user_id:token":"user_id"})

In [196]:
train_pop_per_user = dataset_artists.merge(artist_exposures[["artist","train_popularity"]], left_on="artist", right_on="artist", how="left")[["user_id","train_popularity"]].groupby("user_id").mean()

In [199]:
user_statistics_all_iters["train_pop"] = train_pop_per_user

In [208]:
temp_gender_proportions_df = dataset_artists.merge(artist_exposures[["artist","gender"]], left_on="artist", right_on="artist", how="left")

In [212]:
for gender in temp_gender_proportions_df["gender"].unique():
    temp_gender_proportions_df[str(gender)] = temp_gender_proportions_df["gender"] == gender

In [215]:
temp_gender_proportions_df = temp_gender_proportions_df.drop(columns=["artist"])

In [231]:
gender_proportions_df = temp_gender_proportions_df.groupby('user_id').agg({
    'Male': 'mean',
    'Female': 'mean',
    'Non-binary': 'mean',
    'Other': 'mean'}).reset_index()

In [ ]:
# Get unique countries from tracks
unique_countries = accepted_songs['artist_country'].unique()

# Create one-hot encoding for countries
temp_country_proportions_df = pd.concat([
    accepted_songs[['user_id', 'artist_country']], 
    pd.get_dummies(accepted_songs['artist_country'], prefix='')
], axis=1)

# Group by user and get mean (proportions) - select only numeric columns
country_proportions_df = temp_country_proportions_df.groupby('user_id')[temp_country_proportions_df.columns.difference(['user_id', 'artist_country'])].mean().reset_index()
country_proportions_df

,user_id,country_AU,country_BR,country_CA,country_CO,country_DE,country_DK,country_ES,country_FR,country_IE,...,country_JP,country_NG,country_NL,country_NZ,country_PL,country_SE,country_UK,country_US,country_UY,country_VE
0,1,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.1,0.0,0.0,0.0,0.0,0.0,0.2,0.5,0.0,0.0
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.7,0.0,0.0
2,3,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.6,0.0,0.0
3,4,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.1,0.0,0.1,0.2,0.5,0.0,0.0
4,5,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.8,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
877,935,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.1,0.0,...,0.0,0.0,0.0,0.0,0.0,0.1,0.1,0.6,0.0,0.0
878,936,0.1,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.1,0.0,0.1,0.1,0.5,0.0,0.0
879,937,0.0,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.1,0.3,0.4,0.0,0.0
880,938,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.5,0.0,0.0


# Metrics

## Average Popularity

In [160]:
def average_pop(accepted_songs, artist_exposures):
    average_pop = accepted_songs[["user_id","artist"]].merge(artist_exposures[["artist","train_popularity"]], on="artist", how="left")[["user_id","train_popularity"]].groupby("user_id").mean().rename(columns={"train_popularity":"mean_train_popularity"})
    return average_pop

In [163]:
average_pop(accepted_songs, artist_exposures)

,mean_train_popularity
user_id,
1,0.017874
2,0.015390
3,0.005322
4,0.004836
5,0.008071
...,...
935,0.013549
936,0.005069
937,0.007719
